# FRAUD DETECTION 

A classic problem in financial environments. We will take a look at classifying transitions into fraud and not fraud using scikit-learn package.

This notebook will be using a synthetic database found on kaggle for educational purposes because we all need practice :-).   
The data can be found here: https://www.kaggle.com/datasets/jayjoshi37/digital-payment-fraud-detection/data

I will break this process down into the follwoing steps:

1. Data Exploration 
2. Data prepartion and pre-processing (including feature engineering)
3. Modelling 
4. Evaluation and testing 

We have the following fields within our dataset: 

transaction_id - identifier to each transaction.   
user_id - identifier for each user.   
transaction_amount - amount for each transaction.   
transaction_type - how funds were exchanged e.g "payment" or "bank transfer".   
payment_mode - wallet, card, UPI etc.   
device_type - device transaction was made from e.g iOS.   
device_location - location of the device used to make transaction.   
account_age_days - age of the account.     
transaction_hour - time of transaction in 24 hour notation.    
previous_failed_attempts - if there were previous attempts to make fraudulent transactions.   
avg_transaction_amount - avg amount each account usually makes.   
is_international - is the trasnaction international.    
ip_risk_score - a numerical value  that quantifies the likelihood an IP address is involved in malicious activity, such as fraud, spam, or cyberattacks. 
login_attempts_last_24h - number of login attempts to the account in the last 24 hours.   
fraud_label- is the transaction fraud or not.


## Data Exploration 

In [ ]:
# First we need to start by acquiring all of our dependecies 

# Basic dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt         # visualisations 
from pathlib import Path                # navigating document structure 

# Pipeline
from sklearn.pipeline import Pipeline                       # creating a data pre-propcessing pipeline for our test set 
from sklearn.preprocessing import FunctionTransformer       # to create feature engineering functions 
from sklearn.base import TransformerMixin, BaseEstimator    # used to fit and transform data based on our class 
from sklearn.compose import ColumnTransformer               # for one hot encoding and scaling 
from sklearn.preprocessing import OneHotEncoder, StandardScaler # for one hot encoding and scaling

# Models 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# Metrics 
from sklearn.metrics import accuracy_score,balanced_accuracy_score, precision_recall_curve
from sklearn.metrics import precision_score, recall_score, ConfusionMatrixDisplay # displaying confusion matrix 
from sklearn.metrics import roc_auc_score, roc_curve, auc, confusion_matrix, f1_score
from sklearn.metrics import PrecisionRecallDisplay # displaying AUC 

# Resampling 
from imblearn.over_sampling import SMOTE # for creating synthetic balanced fraud and non fraud cases 
from imblearn.combine import SMOTEENN    # 
from imblearn.pipeline import Pipeline as Pipe
from imblearn.under_sampling import EditedNearestNeighbours
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
# Now we will load our data into our notebook for modelling:
current_dir = Path.cwd()
print(current_dir)              # this directory is 1 level too low in the project structure
parent_dir = current_dir.parent # go up 1 level in the structure 
print(parent_dir)               # now we are in the parent directory that we can now use to naviagte into the dataset 
file_path = parent_dir/ "dataset" / "train_data.csv" 
print(file_path)                # now we are in the correct bracnh of the tree where our dataset is

train = pd.read_csv(file_path)  # IT WORKED!!!


In [ ]:
train.keys()

In [ ]:
train['fraud'].value_counts(normalize=True)

In [ ]:
# Now we will separte the dat into independent variables  x (e.g device location) and dependent variables y (fraud)

x = train[['user_id', 'transaction_amount', 'device_location', 'account_age_days',
       'transaction_hour', 'previous_failed_attempts', 'payment_mode',
       'device_type', 'transaction_type', 'avg_transaction_amount',
       'is_international', 'ip_risk_score', 'login_attempts_last_24h']]
y = train[['fraud']]

In [ ]:
x.nunique()

Our variables with categorical information being: transaction_type, payment_mode, device_type and device_location do not have too many unique values.  
This infroms what methods we can use to deal with these values for classification models that only use numerical data.  
- A possible solution would be to use dummy variables. 

##  DATA PREPARATION AND PRE-PROCESSING 


### PIPELINES & FEATURE ENGINEERING   
We will now create features that improve the performance of our models based on the EDA we conducted in R and the insights we gathered and then create a pipeline to automate the process. 
These are methods we can use to automate the data scince process by doing things like ETL, Pre-processing, Feature engineering and much more. 

We will create our own class to carry out transfromations on the train data d then fit our model to the test data.   
This will include the use of:   
 - FunctionTransformer
 - One hot encoding (replacing dummy variables in a previous iteration)
 - Standrad scaler so values like transaction amounts don't skew our model 

Let's get into it!!

### PIPELINE #1

In [ ]:
#Defining the class
class ETL(BaseEstimator, TransformerMixin): # scikit-learn's way of allowing us to do model training to find pattern and fitting those training weights and baises to data. 
    def __init__(self, alpha = 1): #alpha is used for smoothing our probability value to prevent unknown users from get a rarity of nan.
        self.alpha = alpha
        self.user_rarity_ = None       # user-level probabilities
        self.global_rarity_ = None     # global probabilities

    def fit(self, x , y = None):       # our fit function finds the historical infromation that we will store in a dictionary to transform our test data

        X = x.copy() 
        # Creating user level lookup table 
        # Let us start by  making an empty dataframe we will use to record the prevalnce of each User ID device location 
        users_location_history = pd.DataFrame({ #we have an empty list for each users location history of transactions 
            "user_id" :[],
            "Bangalore" :[], # probability that the user has transaction in Bangalore given thier past history (conditional). 
            "Chennai" :[],   # same as above 
            "Delhi" :[],
            "Hyderabad" :[],
            "Mumbai" :[],
        })   
        users_location_history["user_id"] = X["user_id"].unique()     # this makes an empty dataframe of all of the users we have in the training set 
     

        columns = users_location_history[['Bangalore','Chennai','Delhi','Hyderabad','Mumbai']] #columns where we want to assign zero to contents 
        col_array = np.array(columns)   #convert to array with nan entries 

        nan_array = np.isnan(col_array) # finding all cells that are NaN in col_array
        col_array[nan_array] = 0        # if a cell in array col_array is NaN assign 0
        users_location_history[['Bangalore','Chennai','Delhi','Hyderabad','Mumbai']] = col_array # now all the empty cells have been replaced with zeros          

        
        # Now we get the dummies for each location 
        location_dummies = pd.get_dummies(X, columns = ['device_location'], dtype = int)
        location_dummies = location_dummies[['user_id','device_location_Bangalore'	,                   # ensuring the dummy table is only the device locations and user IDs
                                            'device_location_Chennai',	'device_location_Delhi',	
                                            'device_location_Hyderabad'	,'device_location_Mumbai']]


        location_dummies = location_dummies.groupby(['user_id']).agg({'device_location_Bangalore': 'sum', # getting the sum of locations for each user's historical transactions
                                                                    'device_location_Chennai': 'sum',
                                                                    'device_location_Delhi': 'sum',
                                                                    'device_location_Hyderabad': 'sum',
                                                                    'device_location_Mumbai': 'sum'})

        # USER LEVEL RARITIES - for knonw user transaction histories 
        col_array_prob = np.array(location_dummies[['device_location_Bangalore','device_location_Chennai', # converting this into an array for faster computation
                                                    'device_location_Delhi','device_location_Hyderabad',	
                                                    'device_location_Mumbai']])

        num_features = location_dummies.shape[1]        # number of features in the array (5)
        total_per_user = col_array_prob.sum(axis = 1)   # sum of locations for each user (summation across rows)
        prob_array = (1- (col_array_prob + self.alpha)/(total_per_user[:, None]+ self.alpha * num_features)) #getting the rarity of a user having a transaction in a specific location (the higher the more rare the location for that user)
        users_location_history[['Bangalore','Chennai','Delhi','Hyderabad','Mumbai']] = prob_array # assigning this array to the dataframe 
        users_index = users_location_history.set_index("user_id") # new dataframe with user id as index
        stacked_users_location_history = users_index.stack()      # stacking the infromation from the cells of the dataframe 
        self.user_rarity_ = stacked_users_location_history.to_dict()    # string this data as a dictionary for later 


        #GLOBAL RARITIES - for any new users from the test set 
        total_in_data = col_array_prob.sum(axis=0) # total number of transactions in each location 
        total_across = col_array_prob.sum()        # total number of transactions
        total_prob_array = (1-(total_in_data + self.alpha)/ (total_across + self.alpha * num_features))   # rarities for each location in the training data to be applied to new users 
        city_names = ['Bangalore', 'Chennai', 'Delhi','Hyderabad','Mumbai']      # names of cities for lookups 
        self.global_rarity_ = dict(zip(city_names, total_prob_array))            #saving these rarities as a dictionary  

        return self


    def transform( self, x):        # our transform function can now take the tools we calculated from fit to transform the test data and make predicitions

        X = x.copy() # creating transformations on a copy of the test set 

        user_ids = X["user_id"]             # creating lits of all user ids in the test set 
        locations = X["device_location"]    # creating lits of all device locations in the test set

        x["location_rarity"] = [            # creating our location rarity column by looking up the values in the dictionaries defined above with user id and location as keys
            self.user_rarity_.get((u,l), self.global_rarity_.get(l,self.alpha)) # get the user location rarity from the key value pair in the user rarity dictionary 
            for u,l in zip(user_ids, locations)                                 # if user is a new user ID get global location rarity for that location 
        ]

        return self.drop_features(x)    #return x with dropped features 

    def drop_features(self, x):                           # drop features that we used to create our new features like "user id"
        columns_to_drop = ['user_id','payment_mode', 'device_type', 'transaction_type', 'device_location']
        existing_columns = [ c for c in columns_to_drop if c in x.columns]
        x = x.drop(columns = existing_columns)
        return x

for reference these were some features I made before the ones you see. They were not predcitive at all and had very low balanced accruacy (50%),so it would classify everything as non fraud or high recall (90%) but low balaced accuracy and precision (6%) so it would classify everything as fraud.  This happened no matter the resampling or model used.

That just goes to  how the adage is true: "garbage in garbage out". Bad features go in, bad predictions come out. I'M SO PROUD OF MYSELF!!!

I'm just leaving this here to show my progress. If youre reading this follow me on github and chekc me out: https://github.com/leta199


def location_tag(x):
        #creating the location buckets 
        fraud_location = {  
        "Mumbai":5,     # highest likelihood of fraud 
        "Hyderabad" :4, # second highest likelihood of fraud etc
        "Chennai" : 3,
        "Bangalore" : 2,
        "Delhi" : 1}
        x["fraud_bucket_location"] = x["device_location"].map(fraud_location)
        return x #returning our processed test dataframe 
 
    def risk_buckets(x):
        #creating our risk buckets 
        x["ip_risk_bucket"] = np.where((x['ip_risk_score'] < 0.51) | 
                                        (x['ip_risk_score'] > 0.86), 1, 0)        # assign a 1 if the risk is in the dangerous category and 0 itherwise 

        #creating the account age buckets 
        x["account_age_bucket"] = np.where((x['account_age_days'] >= 1190 ) &
                                            (x['account_age_days'] <= 1855), 1,0) # assign a 1 if we have an account in the most denagrous range and 0 otherwise
        return x
    
    def amount_averages(x):
        transaction_average = pd.DataFrame()                        # empty dataframe to store averages 

        transaction_average["user_id"] = x['user_id'].unique()      # this represents each users average transaction amount for this round of transactions 

        means = x.groupby(['user_id'])['transaction_amount'].mean() # get the mean of each transaction amount from the test data for each User ID

        transaction_average["days_average"] = transaction_average["user_id"].map(means) # add this mean called "days_average" to the dataframe 

        averages_per_user = x.groupby(['user_id'])["avg_transaction_amount"].unique()   # conduct the same procedure for the historic known average of User IDs

        typical_average = averages_per_user.apply(np.mean)                              # get the mean of each average since some User IDs have multiple historic averages 

        transaction_average["typical_average"] = transaction_average["user_id"].map(typical_average) # add this updated historic average to the dataframe 
        x = x.merge(transaction_average, on = "user_id", how = 'left')                               # add this dataframe to x using joins and duplicate some for each User ID so left join
        return x
    
    def transaction_outliers(x):
        days_sd = np.std(x['days_average']) # standard deviation for the days average column (transaction amounts follow roughly uniform distribution form the EDA)

        outlier_days_avg = pd.DataFrame()   # empty dataframe to get the outlier transactions 

        outlier_days_avg["outlier_days_average"] = np.where( (x['transaction_amount'] <= x['days_average'] -  2.5* days_sd) | 
            (x['transaction_amount'] >= x['days_average'] + 2.5 * days_sd), 1,0)           #outliers on days average are transaction amounts outside 2.5 standard deviations from the mean
        
        typical_sd = np.std(x['typical_average']) # similar to the above 

        outlier_typical_avg = pd.DataFrame()
        outlier_typical_avg["outlier_typical_average"] = np.where( (x['transaction_amount'] <= x['days_average'] -  2.5* typical_sd) | 
            (x['transaction_amount'] >= x['days_average'] + 2.5 * typical_sd), 1,0)         # outliers outside 2 standard deviations from the mean are flagged with 1 otherwise 0
        
        x = pd.concat([x,outlier_days_avg,outlier_typical_avg ], axis =1)                   # adding the outliers in terms of the aveage amounts to the test dataframe x
        return x
    
    def international_outliers(x):
        commom_international = x.groupby('user_id')['is_international'].agg(lambda x: pd.Series.mode(x)[0]) # get the most common location (domestric or international) where each User does transactions from
        x['transaction_usual_international'] = x["user_id"].map(commom_international)                       # add this mode to the test dataframe
        x["unexpected_location"] = np.where((x['is_international'] == x['transaction_usual_international'] ) , 0,1) # assign 0 if the transaction is in the usual location and 1 otherwise
        return x

    def login_outliers(x):
        usual_login_attempts = x.groupby('user_id')['previous_failed_attempts'].mean()          # mean of the number of number of attempts to login by User ID 
        x['usual_login_attempts'] = x["user_id"].map(usual_login_attempts)                      # map these onto the test dataframe 
        x["unexpected_login_attempts"] = np.where((x['previous_failed_attempts'] <= x['usual_login_attempts'] ) , 0,1) #assign 0 is the transactions attempts are less than or equal to the normla amount and 1 otherwise 
        return x

In [ ]:
#scaling and get dummies (one hot encoding our variables)
numeric_features = ['transaction_amount', 'account_age_days',
                     'transaction_hour', 'ip_risk_score', ] 
    
categorical_features = [ 'previous_failed_attempts',  'is_international', 'login_attempts_last_24h']

In [ ]:

# We will implore Standardscaler, Onehotencoder and Column transformer to deal with the numeric and categorical features above. 

numeric_transformation = StandardScaler()       # pass numeric features through a scaler
categorical_transformation = OneHotEncoder()    # pass categrical features through  dummy variable generator 

preprocess_data = ColumnTransformer(    
    transformers= [
        ("num" , numeric_transformation, numeric_features),
        ("cat", categorical_transformation,categorical_features)
    ], remainder= "passthrough")                # allow other engineered features to come through to the calssification model 

In [ ]:
# Step 1 - Creating our pipelne 
etl = ETL() # creating instance of the ETL class 

pipeline = Pipeline([
    ("fit_transform", etl),                       # stage 1 - conduct the fit and transform from our class 
    ("preprocess", preprocess_data),              # stage 2 - one hot encode and feature scale 
    ("Logistic_regression", LogisticRegression()) # stage 3 - add our classification model 
])

## MODELLING , EVALUATION AND TESTING 
For this section we will combine all three steps as we create newer pipelines we will fit and evaluate them all at once.  
We can now use the metrics mentined earlier like balanced accuracy, precision and recall to judge our models as well as AUC and different visualisations like Matrix displays 

**Precision** - of all the positines identified how many were truly positive?  
**Recall** - of all possible positive instances, how many did the model catch?   
**ROC AUC** - a performance metric for binary classification models that measures how well the model separates positive and negative classes. It plots the True Positive Rate (Sensitivity) against the False Positive Rate (1-Specificity) at various thresholds, with a score of 1.0 indicating perfect discrimination and 0.5 representing random guessing?  

For fraud detection, we value recall as our main metric but we want to have as good a balance as possible to prevent too many non fraud users getting flagged. 
This is beacuse catching every fraudlent trasaction allows for the least damage to our client base as opposed to flagging a genuine transaction as fraud that can be reversed wuth no harm to the user. 

In [ ]:
#We will now use the pipleine on our test data 
pipeline.fit(x, y)

In [ ]:
current_dir = Path.cwd()
print(current_dir)              # this directory is 1 level too low in the project structure
parent_dir = current_dir.parent # go up 1 level in the structure 
print(parent_dir)               # now we are in the parent directory that we can now use to naviagte into the dataset 
file_path = parent_dir/ "dataset" / "test_data.csv" 
print(file_path)                # now we are in the correct bracnh of the tree where our dataset is

test = pd.read_csv(file_path) 

In [ ]:
test.keys() # test set variables 

In [ ]:
xt = test[['user_id', 'transaction_amount', 'device_location', 'account_age_days',
       'transaction_hour', 'previous_failed_attempts', 'payment_mode',
       'device_type', 'transaction_type', 'avg_transaction_amount',
       'is_international', 'ip_risk_score', 'login_attempts_last_24h', ]]
yt = test[['fraud']]

In [ ]:
y_pred_log = pipeline.predict(xt) #making predictions on the train data 


In [ ]:
accuracy_score(yt, y_pred_log) #we have a very high accuracy score of 93.5% (we correctly classify 93.5% of transactions)


In [ ]:
balanced_accuracy_score(yt,y_pred_log) #our balaned accuracy score is very lowe at 50% 
# (we did not classify any fraud transaction correctly so we only got 1/2 of classifications per class correct) 

In [ ]:
precision_score(yt,y_pred_log) # precision is 0% 
# (no transactions were flagged as fraud)

In [ ]:
recall_score(yt,y_pred_log) #recall is 0% 
# (no transactions were flagged as fraud)

In [ ]:
# Graphical display of Confusion matrix of classifications 
cm = confusion_matrix(yt, y_pred_log) # getting the numerical confusion matrix as an array 
cmd = ConfusionMatrixDisplay(cm)      # plotting the confusiom matrix 
cmd.plot()

### Our model just classifies everything as fraud as it can still obtain a very high accuracy doing so at teh cost of balanced accuracy, precision and recall 

Therefore, we need a method to add greater pressure to our model on the minority (fraud class). 

### PIPELINE #2 (BALANCED CLASS WEIGHTS)

In [ ]:
fit_transform = ETL()

pipeline2 = Pipeline([
    ("fit_transform", etl),
    ("preprocess", preprocess_data),
    ("Logistic_regression", LogisticRegression(class_weight="balanced")) # tells our model that it must put equal emphasis in correclty classifying both classes 
])

In [ ]:
#We will now us the pipleine on our test data with balanced class weights 
pipeline2.fit(x, y)

In [ ]:
y_pred_log_2 = pipeline2.predict(xt) #making predictions on the train data 



In [ ]:
accuracy_score(yt, y_pred_log_2) #we have a very low accuracy of 51.7%


In [ ]:
balanced_accuracy_score(yt,y_pred_log_2) #our balaned accuracy score is slightly higher at 50.1% 

In [ ]:
precision_score(yt,y_pred_log_2) # precision is 6.6% 
# we are now beginning to classify some transactions as fraud

In [ ]:
recall_score(yt,y_pred_log_2) #recall is 48.3%
# we are now catching 48.3% of fraud cases with just one engineered feature

In [ ]:
# Graphical display of Confusion matrix of classifications  with balanced class weights 
cm = confusion_matrix(yt, y_pred_log_2)
cmd = ConfusionMatrixDisplay(cm)
cmd.plot()

In [ ]:
# ROC curve with balanced class wights in pipeline 1 (pipeline2)

y_pred_log_2_roc = pipeline2.predict_proba(xt)[:,1] # get the array of predictions as different probabilities 

fpr, tpr, thresholds = roc_curve(yt, y_pred_log_2_roc) # get the values of false positiev, true positive and thresholds for each 
roc_score = roc_auc_score(yt, y_pred_log)

# Graphical display of the ROC curve 
plt.figure()
plt.plot(fpr, tpr, color='blue', label='ROC curve (area = %0.2f)' % roc_score)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Guessing (AUC = 0.5)') # Diagonal line for random guessing
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()
print(f"The AUC score is: {roc_score}") #This points to our predictions not being much better than random guessing 

### PIPELINE #3 (MORE RARITIES)

For this pipeline we will experiment with using other categorical variables like payment mode to make rarity features and see how they affect our model.

In [ ]:
#Defining the class
class ETL2(BaseEstimator, TransformerMixin): # scikit-learn's way of allowing us to do model training to find pattern and fitting those training weights and baises to data. 
    def __init__(self, alpha = 1, cols = ["device_location", "payment_mode"]): #alpha is used for smoothing our probability value to prevent unknown users from get a rarity of nan.
        self.alpha = alpha
        self.cols = cols
        self.user_rarity_ = {}      # dictionary of dictionaries user-level probabilities
        self.global_rarity_ = {}    # dictionary of dictionaries global probabilities

    def fit(self, x , y = None):    # our fit function finds the historical infromation that we will store in a dictionary to transform our test data

        X = x.copy() # make a copy of our data for transformation 
       
        for col in self.cols:       # for loops that calculates rarity for every column from cols 
            dummies = pd.get_dummies(X[[col, "user_id"]], columns= [col], dtype = int) # select device location and user id and get dummies of device location 
            colnames = [c.replace(f"{col}_", "") for c in dummies.columns if c != "user_id"] # replacing column names in dummies dataframe with empty strings 
            dummies.columns = [c.replace(f"{col}_", "") if c != "user_id" else c for c in dummies.columns] 

            user_stats = dummies.groupby("user_id").sum() # get the aggregate of dummy variables by user id 
            col_array = user_stats.values                 # cast those values to an array 
            total_per_user = col_array.sum(axis = 1)      # get the sum across rows for each user id 

            user_rarity = (1- (col_array + self.alpha)/(total_per_user[:, None]+ self.alpha * col_array.shape[1])) # calucalte rarity metric 

            self.user_rarity_[col] = pd.DataFrame( # assign the rarities to a dataframe, convert to a list and assign it to the user_rarities
                user_rarity, index= user_stats.index, columns = user_stats.columns
            ).stack().to_dict()



        #GLOBAL RARITIES - for any new users from the test set
            total_in_data = col_array.sum(axis=0) # total number of transactions in each each unique identifier in cols  
            total_across = col_array.sum()        # total number of transactions in cols
            total_rarity = (1-(total_in_data + self.alpha)/ (total_across + self.alpha * col_array.shape[1]))   # rarities for each location in the training data to be applied to new users 
            self.global_rarity_[col] = dict(zip(user_stats.columns, total_rarity))                       #saving these rarities as a dictionary  
            
        return self

    def transform( self, x):      # our transform function can now take the tools we calculated from fit to transform the test data and make predicitions
            X = x.copy()

            for col in self.cols: # looping thoruhg all the inputted columns to create new rarity columns in our data 

                user_ids = X["user_id"] # user IDs we want rarity features for 
                feature = X[col]        # list of features we are using to make arraities 

                X[f"{col}_rarity"] = [  # creating a column to store the rarity feature with a name after the column name  
                    self.user_rarity_[col].get((u,l), self.global_rarity_[col].get(l,self.alpha)) # getting the rarity for the column we specified in the loop from the calculated dictionary
                    for u,l in zip(user_ids, feature)
                ]

            return self.drop_features(X)

    def drop_features(self, x):                           # drop features that we used to create our features like "user id"
        columns_to_drop = ['user_id','payment_mode', 'device_type', 'transaction_type', 'device_location']
        existing_columns = [ c for c in columns_to_drop if c in x.columns]
        x = x.drop(columns = existing_columns)
        return x


In [ ]:
 # Using more columns to make rarity features 

etl2 = ETL2(cols = ["device_location", "payment_mode", "transaction_type", "device_type", "is_international"])

pipeline3 = Pipeline([
    ("fit_transform", etl2),
    ("preprocess", preprocess_data),
    ("Logistic_regression", LogisticRegression(class_weight="balanced"))
])

In [ ]:
#We will now us the pipleine on our test data 
pipeline3.fit(x, y)

In [ ]:
y_pred_log_3 = pipeline3.predict(xt) #making predictions on the train data 


In [ ]:
accuracy_score(yt, y_pred_log_3) #we have higher accuracy of 56.4%


In [ ]:
balanced_accuracy_score(yt,y_pred_log_3) # our balaned accuracy score is  low at 50.7% 
# our balanced accuracy has not increased too much from other pipelines 

In [ ]:
precision_score(yt,y_pred_log_3) # precision is 6.7%
# we stil have may false positives 

In [ ]:
recall_score(yt,y_pred_log_3) #recall is 44.2%
# we are catching even more fraud transactions 

In [ ]:
# Graphical display of Confusion matrix of classifications  with balanced class weights and more column rarities 
cm = confusion_matrix(yt, y_pred_log_3)
cmd = ConfusionMatrixDisplay(cm)
cmd.plot()

Coefficients 

We will now out put all of the coefficients to see how our igistic model weighs them 

In [ ]:
# Grab the model from the pipeline
model = pipeline3.named_steps['Logistic_regression']

# Get the feature names after they've been through the ETL and Preprocess steps
# Note: If you have a ColumnTransformer named 'preprocess', we get names from there
feature_names = pipeline3.named_steps['preprocess'].get_feature_names_out()

# Create a clean DataFrame to view them
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_[0]
})

# Sort by absolute value to see the most 'important' features at the top
coef_df['Abs_Weight'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Abs_Weight', ascending=False).drop(columns='Abs_Weight')

print(coef_df)


 - Based on the coefficients we have login attempts of 1 is the most important feature with a coefficient of -0.236 meaning when there is 1 login attempts  the probability of fraud decreases.
 - We also see that international 1 i.e local transactions tend to be less fraudlent and this is the second most  important feature at -0.221

Some other relatively important features are login attempts of 6 (0.172585), international 0 (international transactions) at 0.172378, account age in days (0.151871).

Our rarity features also seem to be pretty prevalent with  device_type_rarity    (-0.121530), then  remainder__transaction_type_rarity   (-0.114492)

Now we can move on to making numerical features and composite features from multiple columns 

In [ ]:
y_pred_log_3_roc = pipeline3.predict_proba(xt)[:,1]

fpr, tpr, thresholds = roc_curve(yt, y_pred_log_3_roc)
roc_score3 = roc_auc_score(yt, y_pred_log)



plt.figure()
plt.plot(fpr, tpr, color='blue', label='ROC curve (area = %0.2f)' % roc_score3)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Guessing (AUC = 0.5)') # Diagonal line for random guessing
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()
print(f"The AUC score is: {roc_score3}") #Our ROC is still not much greater than guessing 

### PIPELINE #4 

This pipeline and round of feature engineering will focus more on creating interactions between numerical features.   
Firstly, we will focus on creating a measure of how much any transaction amount differs from a user's regular patterns.   

This will be the the z-scores of each transcation amount from the user's historical average and group average.  

We will caluclate the standard deviation as the weighted standard deviation from the user and that of their group beginning with 100 groups of 40 people each.  
This is useful to smooth out the standard deviation to account for new user IDs that we dint have historical data for to prevent infinities or NaNs. 

Pro tip - always smooth denominators 

In [ ]:
class NumericETL( BaseEstimator, TransformerMixin):
    def __init__(self, k = 3, n_groups = 100): # initialising our arguments, group size = 100, threshold = 3
        self.k = k                  # a smoothing factor that adijusts weight of group standard deviation vs user's standard deviations (default of 3 means (3/ (no of transaction of user ID +3)) is weight of group std).
        self.n_groups = n_groups    # storing the number of groups 
        self.group_std_ = {}        # empty dictionary to store group standard devations 
        self.user_std_ = {}         # empty dictionary to store group standard devations
        self.global_std_ = []       # list to store historical std of average transactions of all users 
        self.min_ = []              # minimum avg transactions to make cutoff point for groups 
        self.max_ = []              # maximum avg transactions to make cutoff point for groups
        
    def fit(self, x ,y = None):     # our fit function finds the historical infromation that we will store in a dictionary to transform our test data 

        X = x.copy()
        X["groups"] = pd.qcut(X['avg_transaction_amount'], q = self.n_groups , duplicates= 'drop')  # cutting average transaction amounts into 100 groups 
        group_stats = X.groupby(["groups"])['avg_transaction_amount'].std()                         # getting the standard deviation of each group 
        user_std = X.groupby("user_id")["transaction_amount"].std()                                 # getting the standard deviation of each user's transaction amount 

        self.min_ = min(X["avg_transaction_amount"])         # minimum average transaction amount
        self.max_ = max(X["avg_transaction_amount"])         # maximum average transaction amount
        self.group_std_ = group_stats.to_dict()              # saving groups stats to our dictionary 
        self.user_std_ = user_std.to_dict()                  # saving user stats to our dictionary 
        self.global_std_ = X['avg_transaction_amount'].std() # saving the total standard deviation of all average transaction amounts
        return self # must always return self as these are tools we will use to transform the new data 
    

    def transform(self, x):         # our transform function can now take the tools we calculated from fit to transform the test data and make predicitions

        X = x.copy()    
        bin_min = self.min_         # assinging our instance attribute to a local variable 
        bin_max = self.max_         # assinging our instance attribute to a local variable 

        temp_std_user = X["user_id"].map(self.user_std_).fillna(0)              # map our user standard deviation attributes to the user ids we have in our test data, if no historical data then add 0 
        num_transactions = X.groupby("user_id")["user_id"].transform("count")   # the total number of transactions each user ID has in the test data 
        X["avg_transaction_amount"] = X["avg_transaction_amount"].clip(lower = bin_min, upper =bin_max) # clipping all of our average transaction amounts to out bins to apply grouping we calculated
        temp_group_std = X["avg_transaction_amount"].map(self.group_std_).fillna(self.global_std_)      # mapping each average transaction amout above to a group and addign each user a group std

        weight = num_transactions/(num_transactions + self.k)           # weight for user standad deviation 

        X["z_score"] = ((X["transaction_amount"] - X["avg_transaction_amount"])/ # z score for each user's transaction amount 
        (weight*temp_std_user + (1- weight)*temp_group_std))                 # total standard deviation 

        return X        # return tarnsformed data 
        
 

        



In [ ]:
# Pipeline for our numeric feature engineering 

etl4 = NumericETL()  # instance of our NumericETL class 

numeric_features4 = ['transaction_amount', 'account_age_days', # features to scale 
                     'transaction_hour', 'ip_risk_score'] 
    
categorical_features4 = [ 'previous_failed_attempts',  'is_international', 'login_attempts_last_24h'] #features to one hot encode 

numeric_transformation = StandardScaler()
categorical_transformation = OneHotEncoder()

preprocess_data4 = ColumnTransformer( # our steps in scling and encoding the above features 
    transformers=[
        ("num" , numeric_transformation, numeric_features),
        ("cat", categorical_transformation,categorical_features),
        ("z_score", "passthrough", ["z_score"])
    ], 
    remainder="drop" # This will now only "pass through" what it sees in the transformer above 
)


pipeline4 = Pipeline([
    ("fit_transform", etl4),
    ("preprocess", preprocess_data4),
    ("Logistic_regression", LogisticRegression(class_weight="balanced"))
])

In [ ]:
#We will now us the pipleine on our test data 
pipeline4.fit(x, y)

In [ ]:
y_pred_log_4 = pipeline4.predict(xt) #making predictions on the train data 


In [ ]:
accuracy_score(yt, y_pred_log_4) #we have a low accuracy score of 48.9%


In [ ]:
balanced_accuracy_score(yt,y_pred_log_4) #our balaned accuracy score is low at at 50.5%

In [ ]:
precision_score(yt,y_pred_log_4) # precision is 6.7%

In [ ]:
recall_score(yt,y_pred_log_4) #recall is 52.4%

In [ ]:
# Graphical display of Confusion matrix of classifications  with balanced class weights and nuemerical features 
cm = confusion_matrix(yt, y_pred_log_4)
cmd = ConfusionMatrixDisplay(cm)
cmd.plot()

In [ ]:
# Grab the model from the pipeline
model = pipeline4.named_steps['Logistic_regression']

# Get the feature names after they've been through the ETL and Preprocess steps
# Note: If you have a ColumnTransformer named 'preprocess', we get names from there
feature_names = pipeline4.named_steps['preprocess'].get_feature_names_out()

# Create a clean DataFrame to view them
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_[0]
})

# Sort by absolute value to see the most 'important' features at the top
coef_df['Abs_Weight'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Abs_Weight', ascending=False).drop(columns='Abs_Weight')

print(coef_df)

In [ ]:
y_pred_log_4_roc = pipeline4.predict_proba(xt)[:,1]

fpr, tpr, thresholds = roc_curve(yt, y_pred_log_4_roc)
roc_score4 = roc_auc_score(yt, y_pred_log)



plt.figure()
plt.plot(fpr, tpr, color='blue', label='ROC curve (area = %0.2f)' % roc_score4)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Guessing (AUC = 0.5)') # Diagonal line for random guessing
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()
print(f"The AUC score is: {roc_score4}") # Still not too great 

### PIPELINE #5

This pipeline and round of feature engineering will focus more on creating interactions between even more numerical features.   

In [ ]:

class NumericETLmod( BaseEstimator, TransformerMixin):
    def __init__(self, cutoff = 3, aggr_smooth = 1, cpf_smooth = 1, n_groups = 100): # we have changed k to cutoff

        self.cutoff = cutoff                # same as k above 
        self.n_groups = n_groups            # same as above
        self.aggr_smooth = aggr_smooth      # smoothing factor for calculations involving account age 
        self.cpf_smooth = cpf_smooth        # smoothing factor for calculations involving previous failed attempts

        self.group_std_ = {}                # the fit step is the same as that for pipeline 4 NumericETL (refer to previous class)
        self.user_std_ = {}
        
        self.global_std_ = []
        self.min_ = []
        self.max_ = []
        
    def fit(self, x ,y = None):

        X = x.copy()
        X["groups"] = pd.qcut(X['avg_transaction_amount'], q = self.n_groups , duplicates= 'drop')

        group_stats = X.groupby(["groups"])['avg_transaction_amount'].std()
        user_std = X.groupby("user_id")["transaction_amount"].std()

        self.min_ = min(X["avg_transaction_amount"])
        self.max_ = max(X["avg_transaction_amount"])
        self.group_std_ = group_stats.to_dict()
        self.user_std_ = user_std.to_dict()
        self.global_std_ = X['avg_transaction_amount'].std()
        return self 
    

    def transform(self, x):

        X = x.copy() 
        bin_min = self.min_
        bin_max = self.max_

        temp_std_user = X["user_id"].map(self.user_std_).fillna(0)
        num_transactions = X.groupby("user_id")["user_id"].transform("count")
        X["avg_transaction_amount"] = X["avg_transaction_amount"].clip(lower = bin_min, upper =bin_max)
        temp_group_std = X["avg_transaction_amount"].map(self.group_std_).fillna(self.global_std_)

        weight = num_transactions/(num_transactions + self.cutoff)

        X["z_score"] = ((X["transaction_amount"] - X["avg_transaction_amount"])/
        (weight*temp_std_user + (1- weight)*temp_group_std))

        X["login_aggression"] = X["login_attempts_last_24h"] / (X["account_age_days"] +self.aggr_smooth)        # ratio of login attempts to account age (older accounts with many attempts may indicate attempted break-in)
        X["failed_login_aggression"] = X["previous_failed_attempts"]/ (X["account_age_days"] +self.aggr_smooth) # ratio of failed attempts to account age 
        X["failure_rate"] = X["previous_failed_attempts"]/ X["login_attempts_last_24h"]                         # ratio of previous failed attempts to login attempts 
        X["cost_per_failure"] = X["transaction_amount"]/ (X["previous_failed_attempts"] + self.cpf_smooth)      # ratio of transaction amount to previous failed attempts
        X["ATO_Score"] = (X["login_attempts_last_24h"] + X["ip_risk_score"])/ (X["account_age_days"] +self.aggr_smooth) # Account takeover - ratio of ip risk score and login attempts to account age 
        X["ip_age_pressure"] = X["ip_risk_score"]/ (X["account_age_days"] +self.aggr_smooth)                            # ratio of ip risk score to account age 
      


        return X
        

In [ ]:
# Pipeline for our additional numeric features 

etl5 = NumericETLmod()


numeric_transformation = StandardScaler()

preprocess_data5 = ColumnTransformer(
    transformers=[
        ("z_score", "passthrough", ["z_score"]),
        ( "login aggression" ,"passthrough", ["login_aggression"]),
        ("failed_login_aggression", "passthrough", ["failed_login_aggression"]),
        ("failure_rate", "passthrough", ["failure_rate"]),
        ("cost_per_failure", "passthrough", ["cost_per_failure"]),
        ("ATO_Score", "passthrough", ["ATO_Score"]),
        ("ip_age_pressure", "passthrough", ["ip_age_pressure"])
        
    ], 
    remainder="drop" # This will now only "pass through" what it sees in the NEXT fit
)
pipeline5 = Pipeline([
    ("fit_transform", etl5),
    ("preprocess", preprocess_data5),
    ("Logistic_regression", LogisticRegression(class_weight="balanced"))
])

In [ ]:
pipeline5.fit(x,y)

In [ ]:
y_log_pred5 = pipeline5.predict(xt)
y_log_pred5_proba = pipeline5.predict_proba(xt)[:,1]

In [ ]:
balanced_accuracy_score(yt, y_log_pred5) # balanced accuracy slightly higher at 51.5% 
# we will need to look inot finding a way to reduce false positives 


In [ ]:
precision_score(yt, y_log_pred5) # precision of 6.8%
# slightly higher than before but not by much 

In [ ]:
recall_score(yt, y_log_pred5) # recall of 62.6%
# We are managing to cacth more fraud 

In [ ]:
# Graphical display of Confusion matrix of classifications  with balanced class weights and more numerical features 

cm = confusion_matrix(yt, y_log_pred5)
cmd = ConfusionMatrixDisplay(cm)
cmd.plot()

In [ ]:
precision, recall, thresholds = precision_recall_curve(yt,y_log_pred5_proba)

In [ ]:
auc_score = auc(recall, precision)

In [ ]:

display = PrecisionRecallDisplay(precision=precision, recall=recall, average_precision=auc_score)
display.plot()
plt.title(f'Precision-Recall Curve (AUC = {auc_score:.2f})')
plt.show()

# there is a very low AUC meaning that the false poisitive rate is very high compared to true positive 

In [ ]:
# Grab the model from the pipeline
model = pipeline5.named_steps['Logistic_regression']

# Get the feature names after they've been through the ETL and Preprocess steps
# Note: If you have a ColumnTransformer named 'preprocess', we get names from there
feature_names = pipeline5.named_steps['preprocess'].get_feature_names_out()

# Create a clean DataFrame to view them
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': model.coef_[0]
})

# Sort by absolute value to see the most 'important' features at the top
coef_df['Abs_Weight'] = coef_df['Coefficient'].abs()
coef_df = coef_df.sort_values(by='Abs_Weight', ascending=False).drop(columns='Abs_Weight')

print(coef_df)

I made a risk play here by removing all other variables besides the ones we engineered. The rationale was that there may have been multi - collinear features that would compete for the models attention.   

This seemed to be fruitful as now the coefficients of the engineered features were much higher and therefore had greater predictive power. 

### SMOTE 
Synthetic Minority Over-Sampling Technique (SMOTE)

SMOTE is a data-level resampling technique that generates synthetic (artificial) samples for the minority class.   
Instead of simply duplicating existing examples, it creates new data points by interpolating between existing ones.   
This approach allows the model to learn broader patterns and reduces the risk of overfitting to repeated samples.  

Since we have a high false positive rate, we need a better way to distinguish between the true fraud and non fraud.   

We will use **SMOTE ENN** which is a version of SMOTE modified to select n many nearest neighbours that we will keep in the resampling so that it is easier to distinguish between positive and negative classes and the boaundary is harsher in the new sample.   

More information can be found here: https://towardsdatascience.com/imbalanced-classification-in-python-smote-enn-method-db5db06b8d50/

In [ ]:
# Let us begin by adding all of our classes together 

#Defining the class
class ETL_categorical(BaseEstimator, TransformerMixin): # scikit-learn's way of allowing us to do model training to find pattern and fitting those training weights and baises to data. 
    def __init__(self, alpha = 1, cols = ["device_location", "payment_mode"]): #alpha is used for smoothing our probability value to prevent unknown users from get a rarity of nan.
        self.alpha = alpha
        self.cols = cols
        self.user_rarity_ = {}      # dictionary of dictionaries user-level probabilities
        self.global_rarity_ = {}    # dictionary of dictionaries global probabilities

    def fit(self, x , y = None):    # our fit function finds the historical infromation that we will store in a dictionary to transform our test data

        X = x.copy() # make a copy of our data for transformation 
       
        for col in self.cols:       # for loops that calculates rarity for every column from cols 
            dummies = pd.get_dummies(X[[col, "user_id"]], columns= [col], dtype = int) # select device location and user id and get dummies of device location 
            colnames = [c.replace(f"{col}_", "") for c in dummies.columns if c != "user_id"] # replacing column names in dummies dataframe with empty strings 
            dummies.columns = [c.replace(f"{col}_", "") if c != "user_id" else c for c in dummies.columns] 

            user_stats = dummies.groupby("user_id").sum() # get the aggregate of dummy variables by user id 
            col_array = user_stats.values                 # cast those values to an array 
            total_per_user = col_array.sum(axis = 1)      # get the sum across rows for each user id 

            user_rarity = (1- (col_array + self.alpha)/(total_per_user[:, None]+ self.alpha * col_array.shape[1])) # calucalte rarity metric 

            self.user_rarity_[col] = pd.DataFrame( # assign the rarities to a dataframe, convert to a list and assign it to the user_rarities
                user_rarity, index= user_stats.index, columns = user_stats.columns
            ).stack().to_dict()



        #GLOBAL RARITIES - for any new users from the test set
            total_in_data = col_array.sum(axis=0) # total number of transactions in each each unique identifier in cols  
            total_across = col_array.sum()        # total number of transactions in cols
            total_rarity = (1-(total_in_data + self.alpha)/ (total_across + self.alpha * col_array.shape[1]))   # rarities for each location in the training data to be applied to new users 
            self.global_rarity_[col] = dict(zip(user_stats.columns, total_rarity))                       #saving these rarities as a dictionary  
            
        return self

    def transform( self, x):      # our transform function can now take the tools we calculated from fit to transform the test data and make predicitions
            X = x.copy()

            for col in self.cols: # looping through all the inputted columns to create new rarity columns in our data 

                user_ids = X["user_id"] # user IDs we want rarity features for 
                feature = X[col]        # list of features we are using to make arraities 

                X[f"{col}_rarity"] = [  # creating a column to store the rarity feature with a name after the column name  
                    self.user_rarity_[col].get((u,l), self.global_rarity_[col].get(l,self.alpha)) # getting the rarity for the column we specified in the loop from the calculated dictionary
                    for u,l in zip(user_ids, feature)
                ]

            return X
    

class ETL_numeric( BaseEstimator, TransformerMixin):
    def __init__(self, cutoff = 3, aggr_smooth = 1, cpf_smooth = 1, n_groups = 100, std_scaler = 0.1): # we have changed k to cutoff
        
        self.std_scaler = std_scaler
        self.cutoff = cutoff                # same as k above 
        self.n_groups = n_groups            # same as above
        self.aggr_smooth = aggr_smooth      # smoothing factor for calculations involving account age 
        self.cpf_smooth = cpf_smooth        # smoothing factor for calculations involving previous failed attempts

        self.group_std_ = {}                # the fit step is the same as that for pipeline 4 NumericETL (refer to previous class)
        self.user_std_ = {}
        
        self.global_std_ = []
        self.min_ = []
        self.max_ = []
        
    def fit(self, x ,y = None):

        X = x.copy()
        X["groups"] = pd.qcut(X['avg_transaction_amount'], q = self.n_groups , duplicates= 'drop')

        group_stats = X.groupby(["groups"])['avg_transaction_amount'].std()
        user_std = X.groupby("user_id")["transaction_amount"].std()

        self.min_ = min(X["avg_transaction_amount"])
        self.max_ = max(X["avg_transaction_amount"])
        self.group_std_ = group_stats.to_dict()
        self.user_std_ = user_std.to_dict()
        self.global_std_ = X['avg_transaction_amount'].std()
        return self 
    

    def transform(self, x):

        X = x.copy() 
        bin_min = self.min_
        bin_max = self.max_

        temp_std_user = X["user_id"].map(self.user_std_).fillna(0)
        num_transactions = X.groupby("user_id")["user_id"].transform("count")
        X["avg_transaction_amount"] = X["avg_transaction_amount"].clip(lower = bin_min, upper =bin_max)
        temp_group_std = X["avg_transaction_amount"].map(self.group_std_).fillna(self.global_std_)

        weight = num_transactions/(num_transactions + self.cutoff)                                  
        blended_std = np.sqrt((weight*temp_std_user**2 + (1- weight)*temp_group_std**2))        # getting the linear combination of vrainces of the user and group 
        min_std = self.global_std_ * self.std_scaler                                            # minimum allowed standard deviation from global std 
        final_std = np.maximum(blended_std, min_std)                                            # the std we use is the maximum of mimum global std and the blended std
        
        X["transaction_amount_z_score"] = ((X["transaction_amount"] - X["avg_transaction_amount"])/
        final_std)
     
        X["login_aggression"] = X["login_attempts_last_24h"] / (X["account_age_days"] +self.aggr_smooth)        # ratio of login attempts to account age (older accounts with many attempts may indicate attempted break-in)
        X["failed_login_aggression"] = X["previous_failed_attempts"]/ (X["account_age_days"] +self.aggr_smooth) # ratio of failed attempts to account age 
        X["failure_rate"] = X["previous_failed_attempts"]/ X["login_attempts_last_24h"]                         # ratio of previous failed attempts to login attempts 
        X["cost_per_failure"] = X["transaction_amount"]/ (X["previous_failed_attempts"] + self.cpf_smooth)      # ratio of transaction amount to previous failed attempts
        X["ATO_Score"] = (X["login_attempts_last_24h"] + X["ip_risk_score"])/ (X["account_age_days"] +self.aggr_smooth) # Account takeover - ratio of ip risk score and login attempts to account age 
        X["ip_age_pressure"] = X["ip_risk_score"]/ (X["account_age_days"] +self.aggr_smooth)                            # ratio of ip risk score to account age 
        X["trans_amount_average_ratio"] = X["transaction_amount"]/ X["avg_transaction_amount"]
      
        final_data =  self.drop_features(X)
        
        if not hasattr(self, 'fitted_'): 
            print("\n---- Final Dataframe Display -----")
            print(f"Shape of data: {final_data.shape}")
            with pd.option_context('display.expand_frame_repr', False, 'display.max_columns', None):
                print(final_data.head())
            self.fitted_ = True

        return final_data


           
    def drop_features(self, X):                           # drop features that we used to create our features like "user id"
        columns_to_drop = ['user_id',"transaction_amount", 'payment_mode', "account_age",
                           "transaction_hour", "previous_failed_attempts","avg_transaction_amount" ,
                           'device_type', 'transaction_type', 'device_location', "is_international", 
                           "ip_risk_score", "login_attempts_last_24h", "account_age_days"]
        existing_columns = [ c for c in columns_to_drop if c in X.columns]
        X = X.drop(columns = existing_columns)
        return X

We will now create our resampling pipeline. This pipeline conists of: 

1) SMOTE - ENN. 
2) Pipeline creation with imblearn    
3) Fitting the model and evaluation.  
4) Trying out new models  


In [ ]:
# Instancing the SMOTE ENN and our feature transformations 
smote_enn = SMOTE(k_neighbors= 2)
enn_picky = EditedNearestNeighbours( n_neighbors= 1)

sme = SMOTEENN(
    random_state= 123,
    smote = smote_enn,
    enn =  enn_picky
)
etl_categorical = ETL_categorical(cols= ["device_location", "payment_mode", "device_type","transaction_type", "is_international"])
etl_numeric = ETL_numeric()

# Creating the pipeline
pipeline6 = Pipe([
    ("etl_categorical", etl_categorical),
    ("etl_numeric", etl_numeric),
    ("sme", sme),
    ('scaler', StandardScaler()),
    ("logistic_regression", LogisticRegression())
])


In [ ]:
pipeline6.fit(x,y)

In [ ]:
y_pred_log_6 = pipeline6.predict(xt)

In [ ]:
balanced_accuracy_score(yt, y_pred_log_6)

In [ ]:
precision_score(yt, y_pred_log_6)

In [ ]:
recall_score(yt, y_pred_log_6)

In [ ]:
cm_smote = confusion_matrix(yt, y_pred_log_6)
cm_smote_display = ConfusionMatrixDisplay(
    cm_smote
)
cm_smote_display.plot()

In [ ]:
f1_score(yt, y_pred_log_6)

Now that we have created the model with all the features we will implement gridsearchcv to find the best balance of precision and recall 

### GRID SEARCH CV 

In [ ]:
num_features = ["location_rarity" , "device_location_rarity" , "payment_mode_rarity" , "device_type_rarity" ,
                 "transaction_type_rarity" , "is_international_rarity" , "transaction_amount_z_score" ,
                   "login_aggression" , "failed_login_aggression" , "failure_rate"  ,"cost_per_failure" , "ATO_Score",
                   "ip_age_pressure" , "trans_amount_average_ratio"]

numeric_transformation = StandardScaler()

preprocess_data6 = ColumnTransformer( # our steps in scling and encoding the above features 
    transformers=[
        ("num" , numeric_transformation, num_features)
    ] 
)




In [ ]:
pipeline7 = Pipe([
    ("etl_categorical", etl_categorical),
    ("etl_numeric", etl_numeric),
    ("preprocess_data6", preprocess_data6),
    ("sme", sme),
    ("logistic_regression", LogisticRegression())
])


In [ ]:
# Let us begin by creating our parameter grid that our Gridsearchcv will search over 

param_grid = {
    'sme__sampling_strategy' : [0.5,0.9, "auto"],
    "logistic_regression__C" : [0.01,0.1,1,10, 100, 1000, 10000],
    "logistic_regression__penalty" : ['l2'],

    'etl_categorical__alpha'  : [0.5,0.75,1,1.5,2],
    'etl_numeric__cutoff'  : [1,2,3,4,5,6],
    'etl_numeric__aggr_smooth'  : [0.5,0.75,1,1.5],
    'etl_numeric__aggr_smooth'  : [0.5,0.75,1,1.5],
    'etl_numeric__cpf_smooth' : [0.5,0.75,1,1.5],
    'etl_numeric__n_groups'  : [100,200,300,400,500],
    'etl_numeric__std_scaler'  : [0.1,0.2,0.3,0.4],

}
param_grid

In [ ]:
grid_search = RandomizedSearchCV(
    pipeline7, 
    param_distributions= param_grid, 
    n_iter= 50,
    cv=5, 
    scoring='precision',  # Focus on reducing false positives
    n_jobs=-1,
    verbose=2
)

grid_search.fit(x, y)

In [ ]:
print(f"Best Score (F1): {grid_search.best_score_}")
print(f"Best Params: {grid_search.best_params_}")

# Save the best version for testing
best_model = grid_search.best_estimator_

In [ ]:
feature_names = best_model.named_steps['logistic_regression'].feature_names_in_
coefficients = best_model.named_steps['logistic_regression'].coef_[0]

coef_df = pd.DataFrame({'Feature': feature_names, 'Coef': coefficients})
coef_df['Abs_Coef'] = coef_df['Coef'].abs()
print(coef_df.sort_values(by='Abs_Coef', ascending=False))

In [723]:
grid_search.best_params_

{'sme__sampling_strategy': 0.9,
 'logistic_regression__penalty': 'l2',
 'logistic_regression__C': 100,
 'etl_numeric__std_scaler': 0.4,
 'etl_numeric__n_groups': 500,
 'etl_numeric__cutoff': 6,
 'etl_numeric__cpf_smooth': 1,
 'etl_numeric__aggr_smooth': 1.5,
 'etl_categorical__alpha': 0.5}